Używamy modelu ResNet (Mel-spetrogram) na podstawie tekstu MS-SincResNet: Joint learning of 1D and 2D kernels using
multi-scale SincNet and ResNet for music genre classification

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

PROJECT_ROOT = os.getcwd()
CSV_ROOT = os.path.join(PROJECT_ROOT, "csv_files")
IMG_ROOT = os.path.join(PROJECT_ROOT, "DANE_OBRAZOWE_CNN")

IMG_HEIGHT = 256  
IMG_WIDTH  = 256  
BATCH_SIZE = 32   

# Jeśli chcesz trenować osobno, ustaw: "HS" albo "LS"; None = oba
USE_MODALITY = None  # None / "HS" / "LS"

def split_from_csv_name(csv_name: str) -> str:
    # train_segmented.csv -> train
    return os.path.basename(csv_name).split("_")[0]

def png_path_from_row(split: str, row: pd.Series) -> str:
    wav_rel = str(row["filename"])
    base = os.path.splitext(os.path.basename(wav_rel))[0] + ".png"
    modality = str(row["modality"]).strip().upper()
    label = str(int(row["label"]))
    return os.path.join(IMG_ROOT, split, modality, label, base)

def load_split_dataset(csv_name: str, shuffle: bool) -> tf.data.Dataset:
    csv_path = os.path.join(CSV_ROOT, csv_name)
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Nie znaleziono CSV: {csv_path}")

    split = split_from_csv_name(csv_name)
    df = pd.read_csv(csv_path)

    required = {"filename", "label", "modality"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Brak kolumn w {csv_name}: {sorted(missing)}")

    if USE_MODALITY in ("HS", "LS"):
        df = df[df["modality"].astype(str).str.upper() == USE_MODALITY].copy()

    # Ścieżki do PNG + etykiety
    paths = [png_path_from_row(split, r) for _, r in df.iterrows()]
    labels = df["label"].astype(int).to_numpy()

    # Szybka walidacja: ile plików brakuje
    exists_mask = np.array([os.path.exists(p) for p in paths], dtype=bool)
    missing_count = int((~exists_mask).sum())
    if missing_count > 0:
        # zostawiamy tylko te, które istnieją
        paths = list(np.array(paths, dtype=object)[exists_mask])
        labels = labels[exists_mask]
        print(f"[{csv_name}] Brakujących PNG: {missing_count} (pominięte)")

    print(f"[{csv_name}] użyte próbek: {len(paths)} | split={split} | modality={USE_MODALITY or 'HS+LS'}")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(paths), 5000), seed=123, reshuffle_each_iteration=True)

    def _load_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
        img = tf.cast(img, tf.float32)  # rescaling w modelu
        label = tf.cast(label, tf.int32)
        return img, label

    ds = ds.map(_load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

print("Ładowanie danych z CSV -> DANE_OBRAZOWE_CNN...")
train_ds = load_split_dataset("train_segmented.csv", shuffle=True)
val_ds   = load_split_dataset("val_segmented.csv", shuffle=False)
test_ds  = load_split_dataset("test_segmented.csv", shuffle=False)

# MODEL
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

    layers.Conv2D(16, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

#Trening
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

# Test
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)

Ładowanie danych z CSV -> DANE_OBRAZOWE_CNN...
[train_segmented.csv] użyte próbek: 2621 | split=train | modality=HS+LS
[val_segmented.csv] użyte próbek: 372 | split=val | modality=HS+LS
[test_segmented.csv] użyte próbek: 872 | split=test | modality=HS+LS


c:\Users\Karol\Desktop\Analiza_HLS-CMDS\.conda\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 254, 254, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 123008)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │   125,961,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 126,031,969 (480.77 MB)

 Trainable params: 126,031,969 (480.77 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 83s 998ms/step - accuracy: 0.5658 - loss: 1.0643 - val_accuracy: 0.5645 - val_loss: 0.6883
Epoch 2/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.5982 - loss: 0.6692 - val_accuracy: 0.5645 - val_loss: 0.6556
Epoch 3/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 82s 999ms/step - accuracy: 0.7104 - loss: 0.5290 - val_accuracy: 0.7634 - val_loss: 0.5548
Epoch 4/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 82s 996ms/step - accuracy: 0.8642 - loss: 0.2987 - val_accuracy: 0.9220 - val_loss: 0.2102
Epoch 5/10
76/82 ━━━━━━━━━━━━━━━━━━━━ 5s 989ms/step - accuracy: 0.9383 - loss: 0.1614

KeyboardInterrupt: 